In [1]:
import json
import os
import numpy as np
import mne
from typing import List, Tuple, Dict, Optional
import pandas as pd


In [2]:
# ============================================================================
# 全部可调参数（逻辑对齐 ADFTD_preprocessing：窗口 QC + 可选双路 Random Crop 增强）
# ============================================================================

import os

# ---------- 输入路径 ----------
DEMENTIA_JSON_PATH = r"/root/autodl-tmp/CAUEEG Sub1379/dementia.json"
EVENT_DIR = r"/root/autodl-tmp/CAUEEG Sub1379/event"
EDF_DIR = r"/root/autodl-tmp/CAUEEG Sub1379/signal/edf"

# ---------- 输出根目录（子目录名随窗口与增强自动变化，与 ADFTD_preprocessing 一致）----------
output_base_dir = r"/root/autodl-tmp/ADMamba/dataset"

# ---------- 预处理 ----------
TARGET_SAMPLING_RATE = 128.0
BANDPASS_L_FREQ = 0.5
BANDPASS_H_FREQ = 45.0

# ---------- 划窗：WINDOW_TIME（秒）与 STEP_TIME（秒）----------
WINDOW_TIME = 1.0
STEP_TIME = 1.0
WINDOW_SIZE = int(WINDOW_TIME * TARGET_SAMPLING_RATE)
STEP_SAMPLES = max(1, int(STEP_TIME * TARGET_SAMPLING_RATE))
OVERLAP_RATIO = max(0.0, min(0.999, 1.0 - (STEP_SAMPLES / float(WINDOW_SIZE))))
# 是否进行划窗；False 时不切分，每段眼状态序列作为一个样本
ENABLE_WINDOWING = False
# 是否进行窗口质量检测；False 时保留全部窗口（未划窗时则对整段检测）
ENABLE_WINDOW_QC = False

# ---------- Random Cropping 双路增强 ----------
# 关闭划窗时 Random Crop 不生效，两路保存同一整段
USE_AUGMENTATION = False
AUGMENTATION_RANDOM_SEED = 42

# 统一的目录前缀（不含眼状态与增强后缀；默认划窗 + QC 时与原来一致，关闭时追加后缀）
output_tags = []
if not ENABLE_WINDOWING:
    output_tags.append("NoWin")
if not ENABLE_WINDOW_QC:
    output_tags.append("NoQC")
tag_str = ("_" + "_".join(output_tags)) if output_tags else ""
output_dir_name = f"CAUEEG_Dementia_{WINDOW_TIME}s_{OVERLAP_RATIO}{tag_str}"

# 划窗、窗口 QC、增强都关闭时，只保留睁眼 EO
EO_ONLY = (not ENABLE_WINDOWING) and (not ENABLE_WINDOW_QC) and (not USE_AUGMENTATION)

# 同时创建三个输出目录：EC（闭眼）、EO（睁眼）、All（合并）；EO_ONLY 时只创建 EO
if USE_AUGMENTATION:
    output_eo_dir = os.path.join(output_base_dir, f"{output_dir_name}_EO_Aug")
    output_ec_dir = os.path.join(output_base_dir, f"{output_dir_name}_EC_Aug")
    output_all_dir = os.path.join(output_base_dir, f"{output_dir_name}_All_Aug")
else:
    output_eo_dir = os.path.join(output_base_dir, f"{output_dir_name}_EO")
    output_ec_dir = os.path.join(output_base_dir, f"{output_dir_name}_EC")
    output_all_dir = os.path.join(output_base_dir, f"{output_dir_name}_All")

OUTPUT_EO_LABEL_DIR = os.path.join(output_eo_dir, "Label")
OUTPUT_EO_FEATURE_DIR = os.path.join(output_eo_dir, "Feature")
OUTPUT_EC_LABEL_DIR = os.path.join(output_ec_dir, "Label")
OUTPUT_EC_FEATURE_DIR = os.path.join(output_ec_dir, "Feature")
OUTPUT_ALL_LABEL_DIR = os.path.join(output_all_dir, "Label")
OUTPUT_ALL_FEATURE_DIR = os.path.join(output_all_dir, "Feature")

# ---------- 窗口 QC（与 ADFTD 一致；MNE 为伏特，默认换算到微伏）----------
CONVERT_TO_MICROVOLTS = True
QC_BAD_CHANNEL_RATIO_THRESH = 0.2
QC_ABS_AMP_THRESH_UV = 200.0

# ---------- 通道 ----------
KEEP_CHANNELS = [
    "Fp1-AVG", "Fp2-AVG", "F7-AVG", "F3-AVG", "FZ-AVG",
    "F4-AVG", "F8-AVG", "T3-AVG", "C3-AVG", "CZ-AVG",
    "C4-AVG", "T4-AVG", "T5-AVG", "P3-AVG", "PZ-AVG",
    "P4-AVG", "T6-AVG", "O1-AVG", "O2-AVG",
]

os.makedirs(OUTPUT_EO_FEATURE_DIR, exist_ok=True)
os.makedirs(OUTPUT_EO_LABEL_DIR, exist_ok=True)
if not EO_ONLY:
    os.makedirs(OUTPUT_EC_FEATURE_DIR, exist_ok=True)
    os.makedirs(OUTPUT_EC_LABEL_DIR, exist_ok=True)
    os.makedirs(OUTPUT_ALL_FEATURE_DIR, exist_ok=True)
    os.makedirs(OUTPUT_ALL_LABEL_DIR, exist_ok=True)

print("=" * 60)
print("CAUEEG 预处理参数")
print(f"  输出基础目录: {output_base_dir}")
print(f"  输出目录前缀: {output_dir_name}")
print(f"  输出目录(EO): {output_eo_dir}")
if not EO_ONLY:
    print(f"  输出目录(EC): {output_ec_dir}")
    print(f"  输出目录(All): {output_all_dir}")
print(f"  WINDOW_SIZE={WINDOW_SIZE}, STEP_SAMPLES={STEP_SAMPLES}, OVERLAP_RATIO={OVERLAP_RATIO:.4f}")
print(f"  ENABLE_WINDOWING={ENABLE_WINDOWING}, ENABLE_WINDOW_QC={ENABLE_WINDOW_QC}, USE_AUGMENTATION={USE_AUGMENTATION}")
if EO_ONLY:
    print("  只保存 EO：各段睁眼按时间拼接为一条，不检查段长是否一致")
elif USE_AUGMENTATION and not ENABLE_WINDOWING:
    print("  注意: 已关闭划窗，Random Crop 增强不生效，两路保存同一整段")
print(f"  特征目录(EO): {OUTPUT_EO_FEATURE_DIR}")
print(f"  标签目录(EO): {OUTPUT_EO_LABEL_DIR}")
if not EO_ONLY:
    print(f"  特征目录(EC): {OUTPUT_EC_FEATURE_DIR}")
    print(f"  标签目录(EC): {OUTPUT_EC_LABEL_DIR}")
    print(f"  特征目录(All): {OUTPUT_ALL_FEATURE_DIR}")
    print(f"  标签目录(All): {OUTPUT_ALL_LABEL_DIR}")
print("=" * 60)


CAUEEG 预处理参数
  输出基础目录: /root/autodl-tmp/ADMamba/dataset
  输出目录前缀: CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC
  输出目录(EO): /root/autodl-tmp/ADMamba/dataset/CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC_EO
  WINDOW_SIZE=128, STEP_SAMPLES=128, OVERLAP_RATIO=0.0000
  ENABLE_WINDOWING=False, ENABLE_WINDOW_QC=False, USE_AUGMENTATION=False
  只保存 EO：各段睁眼按时间拼接为一条，不检查段长是否一致
  特征目录(EO): /root/autodl-tmp/ADMamba/dataset/CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC_EO/Feature
  标签目录(EO): /root/autodl-tmp/ADMamba/dataset/CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC_EO/Label


In [3]:
def read_json_file(file_path: str) -> Optional[Dict]:
    """读取 JSON 文件并返回其内容"""
    try:
        with open(file_path, 'r') as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"错误：无法读取 {file_path}: {e}")
        return None

In [4]:
def is_valid_pair(event1_name: str, event2_name: str) -> Optional[str]:
    """判断相邻事件对属于 EO/EC 哪种眼状态区间。

    - 当 (Eyes Open -> Eyes Closed) 时，区间是「睁眼 EO」
    - 当 (Eyes Closed -> Eyes Open) 时，区间是「闭眼 EC」
    - 其他事件对返回 None
    """
    if event1_name == "Eyes Open" and event2_name == "Eyes Closed":
        return "EO"
    if event1_name == "Eyes Closed" and event2_name == "Eyes Open":
        return "EC"
    return None

In [5]:
def calculate_eye_event_sample_diffs(events: List[List], sfreq: float, target_sfreq: float) -> List[Tuple[str, int, int, int]]:
    """计算眼状态事件对对应的区间样本差。

    返回的第一个字段为区间类型：
    - "EO": (Eyes Open -> Eyes Closed) 之间的睁眼区间
    - "EC": (Eyes Closed -> Eyes Open) 之间的闭眼区间
    """
    if not events or len(events) < 2:
        return []

    # 调整样本点以适应降采样后的采样率
    scale_factor = sfreq / target_sfreq
    sample_diffs = []
    i = 0
    while i < len(events) - 1:
        current_event = events[i]
        next_event = events[i + 1]
        current_sample, current_name = current_event[0], current_event[1]
        next_sample, next_name = next_event[0], next_event[1]

        # 检查当前事件是否在光刺激区间
        in_photic_block = False
        for j in range(i):
            if events[j][1].startswith("Photic On"):
                photic_off_found = False
                for k in range(j + 1, len(events)):
                    if events[k][1].startswith("Photic Off") and events[k][0] >= events[j][0]:
                        if current_sample >= events[j][0] and current_sample <= events[k][0]:
                            in_photic_block = True
                        photic_off_found = True
                        break
                if not photic_off_found and current_sample >= events[j][0]:
                    in_photic_block = True

        # 如果是有效的事件对且不在光刺激区间，记录时间差
        pair_type = is_valid_pair(current_name, next_name)
        if (not in_photic_block) and (pair_type is not None):
            scaled_current_sample = int(current_sample / scale_factor)
            scaled_next_sample = int(next_sample / scale_factor)
            sample_diff = scaled_next_sample - scaled_current_sample
            sample_diffs.append((pair_type, sample_diff, scaled_current_sample, scaled_next_sample))
            i += 1
        else:
            i += 1
    return sample_diffs

In [6]:
def extract_photic_segments(events: List[List], sfreq: float, target_sfreq: float) -> List[Tuple[int, int]]:
    """提取有效的 Photic On 到 Photic Off 区间，仅包含 Eyes Open/Closed 或无事件，调整样本点以适应降采样"""
    if not events or len(events) < 2:
        return []

    scale_factor = sfreq / target_sfreq
    photic_segments = []
    i = 0
    while i < len(events):
        if events[i][1].startswith("Photic On"):
            photic_on_sample = events[i][0]
            photic_off_sample = None
            valid_segment = True

            # 寻找匹配的 Photic Off
            for j in range(i + 1, len(events)):
                if events[j][1].startswith("Photic Off"):
                    photic_off_sample = events[j][0]
                    break
                # 检查区间内的事件
                if events[j][1] not in ["Eyes Open", "Eyes Closed"]:
                    valid_segment = False
                    break

            if photic_off_sample and valid_segment and photic_off_sample > photic_on_sample:
                scaled_photic_on = int(photic_on_sample / scale_factor)
                scaled_photic_off = int(photic_off_sample / scale_factor)
                photic_segments.append((scaled_photic_on, scaled_photic_off))
            i = j if photic_off_sample else i + 1
        else:
            i += 1
    return photic_segments

In [7]:
def bandpass_filter(raw: mne.io.Raw, l_freq: float = 1.0, h_freq: float = 40.0) -> mne.io.Raw:
    """对 EEG 信号应用带通滤波"""
    try:
        sfreq = raw.info['sfreq']
        if l_freq is not None and l_freq < 0:
            raise ValueError("l_freq 必须 >= 0")
        if h_freq is not None and h_freq >= sfreq / 2:
            h_freq = sfreq / 2 - 0.1
            print(f"警告：h_freq 太高，已自动调整为 {h_freq:.2f} Hz")
        raw_filtered = raw.copy().filter(
            l_freq=l_freq, h_freq=h_freq,
            method='fir', phase='zero', verbose=False
        )
        print(f"应用带通滤波：{l_freq} Hz - {h_freq} Hz")
        return raw_filtered
    except Exception as e:
        print(f"错误：带通滤波失败: {e}")
        return raw

In [8]:
def downsample(raw: mne.io.Raw, target_sfreq: float) -> mne.io.Raw:
    """对 EEG 信号进行降采样"""
    try:
        original_sfreq = raw.info['sfreq']
        if target_sfreq >= original_sfreq:
            print(f"警告：目标采样率 {target_sfreq} Hz 不低于原始采样率 {original_sfreq} Hz，跳过降采样")
            return raw
        raw_downsampled = raw.copy().resample(sfreq=target_sfreq, npad='auto', verbose=False)
        print(f"降采样从 {original_sfreq} Hz 到 {target_sfreq} Hz")
        return raw_downsampled
    except Exception as e:
        print(f"错误：降采样失败: {e}")
        return raw

In [9]:
def window_qc(
    windows,
    sfreq,
    n_channels,
    bad_channel_ratio_thresh=0.2,
    abs_amp_thresh=200.0,
):
    """对窗口进行质量检测（与 ADFTD_preprocessing 一致）。"""
    if windows.ndim != 3:
        raise ValueError(f"windows 必须是 3D 数组，当前形状: {windows.shape}")

    original_shape = windows.shape
    if windows.shape[-1] == n_channels:
        windows = np.transpose(windows, (0, 2, 1))
        print(f"  形状修正: {original_shape} -> {windows.shape} (交换后两个维度)")

    n_w, n_ch, n_t = windows.shape
    ptp = np.ptp(windows, axis=-1)
    ptp_med = np.median(ptp, axis=0)
    ptp_mad = np.median(np.abs(ptp - ptp_med), axis=0) + 1e-6
    ptp_bad = ptp > (ptp_med + 6 * ptp_mad)
    absmax = np.max(np.abs(windows), axis=-1)
    abs_bad = absmax > abs_amp_thresh
    bad_channel_mask = ptp_bad | abs_bad
    bad_ratio = bad_channel_mask.sum(axis=1) / n_ch
    print(f"  坏道数量: {bad_channel_mask.sum(axis=1)}")
    qc_pass = bad_ratio <= bad_channel_ratio_thresh
    return qc_pass


def segment(df, window_size, overlap_ratio=0.0, sfreq=128, enable_qc=True,
            bad_channel_ratio_thresh=0.2, abs_amp_thresh=200.0):
    """窗口分割（中点锚定滑窗）+ 可选 QC，与 ADFTD_preprocessing 一致。"""
    res = []
    data_length = df.shape[0]
    stride = int(window_size * (1 - overlap_ratio))
    start = int(data_length / 2)
    left_index = start - int(start / stride) * stride
    right_index = start + int((data_length - start) / stride) * stride
    for i in range(left_index, right_index, stride):
        if i + window_size <= data_length:
            res.append(df.iloc[i : i + window_size, :])
    if enable_qc and len(res) > 0:
        windows_array = np.array([window.values.T for window in res])
        qc_pass = window_qc(
            windows_array,
            sfreq,
            n_channels=df.shape[1],
            bad_channel_ratio_thresh=bad_channel_ratio_thresh,
            abs_amp_thresh=abs_amp_thresh,
        )
        res_filtered = [res[i] for i in range(len(res)) if qc_pass[i]]
        print(f"  质量控制: 总窗口数 {len(res)}, 通过 {qc_pass.sum()}, 丢弃 {(~qc_pass).sum()}")
        return res_filtered
    return res


def segment_random_crop_pair(
    df,
    window_size,
    overlap_ratio=0.0,
    sfreq=128,
    enable_qc=True,
    bad_channel_ratio_thresh=0.2,
    abs_amp_thresh=200.0,
    rng=None,
    random_seed=42,
):
    """Random Cropping 双路 + 成对 QC，与 ADFTD_preprocessing 一致。"""
    if rng is None:
        rng = np.random.default_rng(random_seed)

    data_length = df.shape[0]
    W = int(window_size)
    stride = max(1, int(round(W * (1 - float(overlap_ratio)))))

    empty_meta = {
        "start_ori": None,
        "start_aug": None,
        "gap": None,
        "n_windows": 0,
        "reserved_tail": None,
    }
    if data_length < W:
        return [], [], empty_meta

    last_start = ((data_length - W) // stride) * stride
    tail = data_length - (last_start + W)
    if tail == 0:
        reserved_tail = W
    elif tail < (W / 2.0):
        reserved_tail = 2 * W
    else:
        reserved_tail = W

    L_work = data_length - reserved_tail
    if L_work < W:
        return [], [], {**empty_meta, "reserved_tail": reserved_tail, "reason": "L_work < W"}

    n_windows = (L_work - W) // stride + 1
    max_start = L_work - W - (n_windows - 1) * stride
    if max_start < 0:
        return [], [], {**empty_meta, "reserved_tail": reserved_tail, "reason": "max_start < 0"}

    d_min = int(np.ceil(W / 10.0))
    d_max = int(np.floor(W / 2.0))
    if d_min > d_max:
        raise ValueError(
            f"window_size={W} 过小：无法满足起点间隔 ∈ [W/10, W/2]（得到 d_min={d_min}, d_max={d_max}）"
        )

    start_ori = None
    start_aug = None
    gap = None
    if max_start == 0:
        start_ori = 0
        start_aug = 0
        gap = 0
    else:
        d_lo, d_hi = d_min, d_max
        if max_start < d_min:
            d_lo = 1
            d_hi = min(d_max, max_start)
        if d_lo > d_hi:
            start_ori = 0
            start_aug = 0
            gap = 0
        else:
            for _ in range(500):
                g = int(rng.integers(d_lo, d_hi + 1))
                upward = rng.random() < 0.5
                if upward:
                    lo, hi = 0, max_start - g
                    if lo > hi:
                        continue
                    start_ori = int(rng.integers(lo, hi + 1))
                    start_aug = start_ori + g
                else:
                    lo, hi = g, max_start
                    if lo > hi:
                        continue
                    start_ori = int(rng.integers(lo, hi + 1))
                    start_aug = start_ori - g
                gap = g
                break

    if start_ori is None:
        raise RuntimeError(
            f"无法在约束下采样到合法的 (start_ori, start_aug)；请检查 window_size、overlap_ratio 与数据长度。L_work={L_work}"
        )

    res_ori = []
    res_aug = []
    for k in range(n_windows):
        i_ori = start_ori + k * stride
        i_aug = start_aug + k * stride
        res_ori.append(df.iloc[i_ori : i_ori + W, :])
        res_aug.append(df.iloc[i_aug : i_aug + W, :])

    meta = {
        "start_ori": start_ori,
        "start_aug": start_aug,
        "gap": gap,
        "n_windows": n_windows,
        "L_work": L_work,
        "stride": stride,
    }

    if enable_qc and n_windows > 0:
        arr_ori = np.array([w.values.T for w in res_ori])
        arr_aug = np.array([w.values.T for w in res_aug])
        qc_ori = window_qc(
            arr_ori,
            sfreq,
            n_channels=df.shape[1],
            bad_channel_ratio_thresh=bad_channel_ratio_thresh,
            abs_amp_thresh=abs_amp_thresh,
        )
        qc_aug = window_qc(
            arr_aug,
            sfreq,
            n_channels=df.shape[1],
            bad_channel_ratio_thresh=bad_channel_ratio_thresh,
            abs_amp_thresh=abs_amp_thresh,
        )
        qc_pair = qc_ori & qc_aug
        res_ori_f = [res_ori[i] for i in range(n_windows) if qc_pair[i]]
        res_aug_f = [res_aug[i] for i in range(n_windows) if qc_pair[i]]
        meta["qc_pass_ori"] = qc_ori
        meta["qc_pass_aug"] = qc_aug
        meta["qc_pass_pair"] = qc_pair
        print(
            f"  [random_crop_pair] QC: 总窗口 {n_windows}, 两路均通过 {qc_pair.sum()}, 丢弃 {n_windows - qc_pair.sum()}"
        )
        return res_ori_f, res_aug_f, meta

    return res_ori, res_aug, meta


In [10]:
def process_eeg_data():
    """同时输出 EO / EC / All 三套数据（目录结构参考 CAEEG_Supervised）。"""
    dementia_data = read_json_file(DEMENTIA_JSON_PATH)
    if not dementia_data:
        print("错误：无法加载 " + DEMENTIA_JSON_PATH)
        return

    all_samples = (
        dementia_data.get("train_split", [])
        + dementia_data.get("validation_split", [])
        + dementia_data.get("test_split", [])
    )
    if not all_samples:
        print("错误：dementia.json 中没有样本数据")
        return

    serial_to_label = {}
    total_eo = 0
    total_ec = 0
    total_all = 0

    for sample in all_samples:
        serial = sample["serial"]
        class_label = sample["class_label"]
        serial_to_label[serial] = class_label
        print(f"处理序列号: {serial}，标签: {class_label}")

        event_file = os.path.join(EVENT_DIR, f"{serial}.json")
        events = read_json_file(event_file)
        if not events:
            print(f"警告：无法读取事件文件 {event_file}，跳过")
            continue

        edf_file = os.path.join(EDF_DIR, f"{serial}.edf")
        try:
            raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
        except Exception as e:
            print(f"错误：无法读取 EDF 文件 {edf_file}: {e}")
            continue

        original_sfreq = raw.info["sfreq"]
        raw.pick(KEEP_CHANNELS, verbose=False)
        raw = bandpass_filter(raw, BANDPASS_L_FREQ, BANDPASS_H_FREQ)
        raw = downsample(raw, TARGET_SAMPLING_RATE)

        sample_diffs = calculate_eye_event_sample_diffs(events, original_sfreq, TARGET_SAMPLING_RATE)
        if not sample_diffs:
            print(f"警告：{serial} 没有有效的眼睛事件对")

        data, _times = raw.get_data(return_times=True)
        del raw

        # EO/EC/All 三套容器
        eo_chunk, ec_chunk, all_chunk = [], [], []
        eo_ori, eo_aug = [], []
        ec_ori, ec_aug = [], []
        all_ori, all_aug = [], []

        for pair_type, sample_diff, start_sample, end_sample in sample_diffs:
            if EO_ONLY and pair_type != "EO":
                continue
            if not ENABLE_WINDOWING:
                if sample_diff <= 0:
                    continue
            elif sample_diff < WINDOW_SIZE:
                continue
            segment_data = data[:, start_sample:end_sample]
            seg = segment_data.astype(np.float64)
            if CONVERT_TO_MICROVOLTS:
                seg = seg * 1e6
            df_seg = pd.DataFrame(seg.T)

            if not ENABLE_WINDOWING:
                frames = [df_seg] if df_seg.shape[0] > 0 else []
                if ENABLE_WINDOW_QC and frames:
                    windows_array = np.array([df_seg.values.T])
                    qc_pass = window_qc(
                        windows_array,
                        TARGET_SAMPLING_RATE,
                        n_channels=df_seg.shape[1],
                        bad_channel_ratio_thresh=QC_BAD_CHANNEL_RATIO_THRESH,
                        abs_amp_thresh=QC_ABS_AMP_THRESH_UV,
                    )
                    kept = bool(qc_pass[0])
                    print(f"  质量控制(整段): {'通过' if kept else '丢弃'}")
                    frames = [df_seg] if kept else []
                if USE_AUGMENTATION:
                    for frame in frames:
                        xo = frame.values.astype(np.float32)
                        xa = xo.copy()
                        if pair_type == "EO":
                            eo_ori.append(xo)
                            eo_aug.append(xa)
                        elif pair_type == "EC":
                            ec_ori.append(xo)
                            ec_aug.append(xa)
                        all_ori.append(xo)
                        all_aug.append(xa)
                else:
                    for frame in frames:
                        xw = frame.values.astype(np.float32)
                        if pair_type == "EO":
                            eo_chunk.append(xw)
                        elif pair_type == "EC":
                            ec_chunk.append(xw)
                        all_chunk.append(xw)
                continue

            if USE_AUGMENTATION:
                res_ori, res_aug, _meta = segment_random_crop_pair(
                    df_seg,
                    WINDOW_SIZE,
                    OVERLAP_RATIO,
                    sfreq=TARGET_SAMPLING_RATE,
                    enable_qc=ENABLE_WINDOW_QC,
                    bad_channel_ratio_thresh=QC_BAD_CHANNEL_RATIO_THRESH,
                    abs_amp_thresh=QC_ABS_AMP_THRESH_UV,
                    random_seed=AUGMENTATION_RANDOM_SEED,
                )
                for df_o, df_a in zip(res_ori, res_aug):
                    xo = df_o.values.astype(np.float32)
                    xa = df_a.values.astype(np.float32)
                    if pair_type == "EO":
                        eo_ori.append(xo)
                        eo_aug.append(xa)
                    elif pair_type == "EC":
                        ec_ori.append(xo)
                        ec_aug.append(xa)
                    all_ori.append(xo)
                    all_aug.append(xa)
            else:
                res_df = segment(
                    df_seg,
                    WINDOW_SIZE,
                    OVERLAP_RATIO,
                    sfreq=TARGET_SAMPLING_RATE,
                    enable_qc=ENABLE_WINDOW_QC,
                    bad_channel_ratio_thresh=QC_BAD_CHANNEL_RATIO_THRESH,
                    abs_amp_thresh=QC_ABS_AMP_THRESH_UV,
                )
                for df_w in res_df:
                    xw = df_w.values.astype(np.float32)
                    if pair_type == "EO":
                        eo_chunk.append(xw)
                    elif pair_type == "EC":
                        ec_chunk.append(xw)
                    all_chunk.append(xw)

        # 保存（每个 serial 可能只有 EO/EC 中的一部分有数据）
        # 三个开关都关闭时只保存 EO，各段睁眼按时间拼接，不要求段长一致
        if EO_ONLY:
            if eo_chunk:
                merged = np.concatenate(eo_chunk, axis=0).astype(np.float32)
                out = merged[np.newaxis, ...]
                eo_path = os.path.join(OUTPUT_EO_FEATURE_DIR, f"feature_{serial}.npy")
                np.save(eo_path, out)
                print(f"  保存 EO: {eo_path} 形状 {out.shape}（拼接 {len(eo_chunk)} 段）")
                total_eo += 1
        else:
            def _same_shape(chunks):
                if isinstance(chunks, np.ndarray) or not chunks:
                    return True
                return len({tuple(np.shape(x)) for x in chunks}) == 1

            def _save_single(feature_dir, arr, n_count, tag):
                if arr is None:
                    return 0
                if isinstance(arr, np.ndarray):
                    out = arr
                else:
                    if not _same_shape(arr):
                        print(f"  警告：{serial} 的 {tag} 各段长度不一致，无法堆叠，跳过")
                        return 0
                    out = np.asarray(arr, dtype=np.float32)
                if out.size == 0:
                    return 0
                np.save(os.path.join(feature_dir, f"feature_{serial}.npy"), out)
                print(f"  保存 {tag}: {os.path.join(feature_dir, f'feature_{serial}.npy')} 形状 {out.shape}")
                return n_count(out)

            if USE_AUGMENTATION:
                def pack(ori, aug):
                    if not ori:
                        return None
                    if not _same_shape(ori) or not _same_shape(aug):
                        print(f"  警告：{serial} 各段长度不一致，无法堆叠，跳过")
                        return None
                    o = np.asarray(ori, dtype=np.float32)
                    a = np.asarray(aug, dtype=np.float32)
                    if o.shape != a.shape:
                        raise ValueError(f"{serial}: 原始/增强窗口形状不一致 {o.shape} vs {a.shape}")
                    return np.stack([o, a], axis=0)

                eo_out = pack(eo_ori, eo_aug)
                ec_out = pack(ec_ori, ec_aug)
                all_out = pack(all_ori, all_aug)

                total_eo += _save_single(OUTPUT_EO_FEATURE_DIR, eo_out, lambda x: int(x.shape[1]), "EO")
                total_ec += _save_single(OUTPUT_EC_FEATURE_DIR, ec_out, lambda x: int(x.shape[1]), "EC")
                total_all += _save_single(OUTPUT_ALL_FEATURE_DIR, all_out, lambda x: int(x.shape[1]), "All")
            else:
                total_eo += _save_single(OUTPUT_EO_FEATURE_DIR, eo_chunk, lambda x: int(x.shape[0]), "EO")
                total_ec += _save_single(OUTPUT_EC_FEATURE_DIR, ec_chunk, lambda x: int(x.shape[0]), "EC")
                total_all += _save_single(OUTPUT_ALL_FEATURE_DIR, all_chunk, lambda x: int(x.shape[0]), "All")

    # 标签按序列号排序；EO_ONLY 时只写入 EO 目录
    if serial_to_label:
        sorted_serials = sorted(serial_to_label.keys())
        labels = np.array(
            [[serial_to_label[s], int(s)] for s in sorted_serials],
            dtype=np.int32,
        )
        np.save(os.path.join(OUTPUT_EO_LABEL_DIR, "label.npy"), labels)
        if not EO_ONLY:
            np.save(os.path.join(OUTPUT_EC_LABEL_DIR, "label.npy"), labels)
            np.save(os.path.join(OUTPUT_ALL_LABEL_DIR, "label.npy"), labels)
        print(f"保存标签：shape={labels.shape}，序列号顺序: {sorted_serials}")
    else:
        print("错误：没有生成任何有效数据或标签")

    if EO_ONLY:
        print(f"总样本数 EO={total_eo}")
    else:
        print(f"总窗口数 EO={total_eo}, EC={total_ec}, All={total_all}（增强模式下按「对」计数）")


In [11]:
if __name__ == "__main__":
    process_eeg_data()


处理序列号: 00587，标签: 0
应用带通滤波：0.5 Hz - 45.0 Hz
降采样从 200.0 Hz 到 128.0 Hz
  保存 EO: /root/autodl-tmp/ADMamba/dataset/CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC_EO/Feature/feature_00587.npy 形状 (1, 35499, 19)（拼接 10 段）
处理序列号: 01301，标签: 2
应用带通滤波：0.5 Hz - 45.0 Hz
降采样从 200.0 Hz 到 128.0 Hz
警告：01301 没有有效的眼睛事件对
处理序列号: 00781，标签: 1
应用带通滤波：0.5 Hz - 45.0 Hz
降采样从 200.0 Hz 到 128.0 Hz
  保存 EO: /root/autodl-tmp/ADMamba/dataset/CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC_EO/Feature/feature_00781.npy 形状 (1, 28444, 19)（拼接 9 段）
处理序列号: 00788，标签: 0
应用带通滤波：0.5 Hz - 45.0 Hz
降采样从 200.0 Hz 到 128.0 Hz
  保存 EO: /root/autodl-tmp/ADMamba/dataset/CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC_EO/Feature/feature_00788.npy 形状 (1, 36719, 19)（拼接 10 段）
处理序列号: 00479，标签: 1
应用带通滤波：0.5 Hz - 45.0 Hz
降采样从 200.0 Hz 到 128.0 Hz
  保存 EO: /root/autodl-tmp/ADMamba/dataset/CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC_EO/Feature/feature_00479.npy 形状 (1, 26130, 19)（拼接 6 段）
处理序列号: 00892，标签: 0
应用带通滤波：0.5 Hz - 45.0 Hz
降采样从 200.0 Hz 到 128.0 Hz
  保存 EO: /root/autodl-tmp/ADMamba/data

In [12]:
# 可选：检查已保存特征形状（需先运行「配置参数」单元）
def count_original_samples(shape):
    """增强 layout (2, N, T, C) 时只统计第 1 维原始样本数 N"""
    if len(shape) == 4 and shape[0] == 2:
        return shape[1]
    return shape[0]

for tag, feat_dir in [("EO", OUTPUT_EO_FEATURE_DIR), ("EC", OUTPUT_EC_FEATURE_DIR), ("All", OUTPUT_ALL_FEATURE_DIR)]:
    print("-" * 40)
    print(tag, feat_dir)
    eo_total_samples = 0
    for file in sorted(os.listdir(feat_dir)):
        sub_path = os.path.join(feat_dir, file)
        shape = np.load(sub_path).shape
        print(file, shape)
        if tag == "EO":
            eo_total_samples += count_original_samples(shape)
    if tag == "EO":
        print(f"EO 原始样本总数: {eo_total_samples}")

for tag, lab_dir in [("EO", OUTPUT_EO_LABEL_DIR), ("EC", OUTPUT_EC_LABEL_DIR), ("All", OUTPUT_ALL_LABEL_DIR)]:
    label_path = os.path.join(lab_dir, "label.npy")
    if os.path.exists(label_path):
        print(tag, "label:", np.load(label_path).shape)


----------------------------------------
EO /root/autodl-tmp/ADMamba/dataset/CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC_EO/Feature
feature_00001.npy (1, 30048, 19)
feature_00002.npy (1, 27579, 19)
feature_00003.npy (1, 2823, 19)
feature_00004.npy (1, 31248, 19)
feature_00005.npy (1, 18922, 19)
feature_00006.npy (1, 26422, 19)
feature_00007.npy (1, 11612, 19)
feature_00008.npy (1, 8067, 19)
feature_00011.npy (1, 36776, 19)
feature_00012.npy (1, 18840, 19)
feature_00013.npy (1, 3454, 19)
feature_00014.npy (1, 27202, 19)
feature_00016.npy (1, 37820, 19)
feature_00017.npy (1, 26348, 19)
feature_00018.npy (1, 29656, 19)
feature_00019.npy (1, 9813, 19)
feature_00020.npy (1, 25374, 19)
feature_00021.npy (1, 38704, 19)
feature_00022.npy (1, 35966, 19)
feature_00024.npy (1, 14774, 19)
feature_00025.npy (1, 34703, 19)
feature_00026.npy (1, 24921, 19)
feature_00027.npy (1, 10589, 19)
feature_00028.npy (1, 12841, 19)
feature_00029.npy (1, 29652, 19)
feature_00030.npy (1, 29437, 19)
feature_00031.npy (1, 

FileNotFoundError: [Errno 2] No such file or directory: '/root/autodl-tmp/ADMamba/dataset/CAUEEG_Dementia_1.0s_0.0_NoWin_NoQC_EC/Feature'